In [1]:
!pip install wikipedia==1.4.0 youtube-search==2.1.2

'pip' is not recognized as an internal or external command,
operable program or batch file.


Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [2]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

# Optional keys. A few notebooks call a third-party API : Tavily in demos05a,
# OpenWeatherMap in demos05b, LangSmith in demos09. load_dotenv() covers the
# local path ; in Colab there is no .env, so they are read from Secrets here.
# Missing is fine, the cell that needs one says so.
if IN_COLAB:
    try:
        from google.colab import userdata
        for _name in ("TAVILY_API_KEY", "OWM_API_KEY", "LANGSMITH_API_KEY"):
            try:
                _v = userdata.get(_name)
                if _v:
                    os.environ[_name] = _v
            except Exception:
                pass
    except ImportError:
        pass


model=qwen3.5:2b


### Simple Agent
The modern way to build one is with prebuilt create_agent, which has replaced the older AgentExecutor approach. 
An agent differs from a chain in one key way: instead of running a fixed sequence of steps, the model itself decides which tools 
to call and when, looping until it has an answer.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# 1. Define tools : plain functions with the @tool decorator
@tool
def desks_free(floor: int) -> int:
    """How many desks are free on a floor today."""
    return {1: 14, 2: 3, 3: 0}.get(floor, 0)

@tool
def room_status(room: str) -> str:
    """Whether a meeting room is free, and until when."""
    return f"{room} is free until 14:00."

tools = [desks_free, room_status]

# 2. The model
llm = make_llm()

# 3. Build the agent
agent = create_agent(llm, tools)

# 4. Invoke : input is a messages dict
result = agent.invoke({"messages": [("human", "How many desks are free on floor 2, and is Room Aurora free?")]})
for message in result["messages"]:
    message.pretty_print()

# The final answer is the last message
print(result["messages"][-1].content)

================================ Human Message =================================

How many desks are free on floor 2, and is Room Aurora free?
================================== Ai Message ==================================
Tool Calls:
  desks_free (call_desuc531)
 Call ID: call_desuc531
  Args:
    floor: 2
  room_status (call_3kpf1vha)
 Call ID: call_3kpf1vha
  Args:
    room: Aurora
================================= Tool Message =================================
Name: desks_free

3
================================= Tool Message =================================
Name: room_status

Aurora is free until 14:00.
================================== Ai Message ==================================

There are **3** desks free on floor 2, and Room Aurora is free until 14:00.
There are **3** desks free on floor 2, and Room Aurora is free until 14:00.


### Agent without Tools
Technically an agent can be created without tools, but it stops being meaningful as an agent. An agent without tools is a model call with extra machinery around it ; the thing that makes an agent an agent is precisely the loop of deciding which tool to call, calling it, a
nd reacting to the result. Take the tools away and there's nothing to decide and no loop to run; the model reads our question 
and produces text in one pass, which is exactly what a plain chain or a direct llm.invoke() already does.

In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = make_llm()

agent = create_agent(llm, [])   # no tools

result = agent.invoke({"messages": [("human", "What is a service level agreement?")]})
print(result["messages"][-1].content)   # answered from training data, no tools involved

A **Service Level Agreement (SLA)** is a formal contract between two parties that defines the quality, availability, and performance standards of a specific service. It acts as a binding promise to deliver a certain level of service within agreed-upon parameters, outlining exactly what happens if those standards are not met.

### Key Components of an SLA
While every SLA is tailored to the industry and client needs, they typically include the following core elements:

*   **Service Description**: A clear definition of what the provider is offering (e.g., IT support, cloud hosting, customer support).
*   **Key Performance Indicators (KPIs)**: Specific metrics used to measure success. Common examples include:
    *   **Uptime/Availability**: The percentage of time the service is operational (e.g., 99.9%).
    *   **Response Times**: How quickly a support agent replies to a ticket (e.g., "within 1 hour").
    *   **Resolution Time**: How long it takes to fix an issue once reported.
*   **P

In [5]:
%pip install -q yt-dlp

Note: you may need to restart the kernel to use updated packages.


### Agent with Wikipedia and YouTube

In [6]:
import warnings
warnings.filterwarnings("ignore")

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import yt_dlp, wikipedia
import re

# LLM : OpenAI-compatible bridge to local Ollama
llm = make_llm()

# --- Tools: each catches its own errors so a bad web response can't crash the agent ---

@tool
def wikipedia_search(query: str) -> str:
    """Look up a topic on Wikipedia and return a short summary."""
    try:
        return wikipedia.summary(query, sentences=5, auto_suggest=False)
    except Exception as e:
        return f"Wikipedia lookup failed: {e}"

@tool
def youtube_search(query: str) -> str:
    """Search YouTube and return the top video link for the query."""
    try:
        opts = {"quiet": True, "extract_flat": True, "skip_download": True}
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(f"ytsearch1:{query}", download=False)
        entries = info.get("entries") or []
        if not entries:
            return "No YouTube results found."
        v = entries[0]
        url = f"https://www.youtube.com/watch?v={v['id']}"
        return f"Video: {v.get('title', '')}\nLink: {url}"
    except Exception as e:
        return f"YouTube search failed: {e}"

agent_tools = [youtube_search, wikipedia_search]
agent = create_agent(llm, agent_tools)

# --- Pull any YouTube links straight from the tool messages, not the model's prose ---

def extract_youtube_links(messages):
    links = []
    for m in messages:
        if isinstance(m, ToolMessage) and m.name == "youtube_search":
            links += re.findall(r"https://www\.youtube\.com/watch\?v=\S+", m.content)
    return links

# --- Conversation memory per thread_id ---

threads = {}

def run_agent(user_text: str, thread_id: str = "srk-demo"):
    if thread_id not in threads:
        threads[thread_id] = []
    messages = threads[thread_id] + [HumanMessage(content=user_text)]
    result = agent.invoke({"messages": messages})
    reply = result["messages"][-1].content

    # Guarantee the link is present, even if the model omitted it
    links = extract_youtube_links(result["messages"])
    for url in links:
        if url not in reply:
            reply += f"\n\nYouTube link: {url}"

    threads[thread_id].append(HumanMessage(content=user_text))
    threads[thread_id].append(AIMessage(content=reply))
    return reply

print(run_agent("Tell me something about Sylvester Stallone. "
                "Also, get me the link to one of his movies from YouTube."))

I apologize, but I was unable to retrieve information about Sylvester Stallone or find a YouTube link for his movies. The tools used in this session encountered errors.

However, I can tell you that **Sylvester Stallone** is an American actor and director best known for his iconic role as Rocky Balboa in the *Rocky* film series (1976–1979). He also starred in the movie *Rambo: First Blood Part II* (1985) and directed the 1993 film *The Rock*.


In [7]:
import langchain.tools as tools
tools.__all__

['BaseTool',
 'InjectedState',
 'InjectedStore',
 'InjectedToolArg',
 'InjectedToolCallId',
 'ToolException',
 'ToolRuntime',
 'tool']

In [8]:
import pkgutil
import langchain_community.agent_toolkits as atk

print([m.name for m in pkgutil.iter_modules(atk.__path__)])

['ainetwork', 'amadeus', 'azure_ai_services', 'azure_cognitive_services', 'base', 'cassandra_database', 'clickup', 'cogniswitch', 'connery', 'csv', 'file_management', 'financial_datasets', 'github', 'gitlab', 'gmail', 'jira', 'json', 'load_tools', 'multion', 'nasa', 'nla', 'office365', 'openapi', 'playwright', 'polygon', 'powerbi', 'slack', 'spark_sql', 'sql', 'steam', 'xorbits', 'zapier']


### OpenWeatherMap API

In [9]:
from typing import Type
from pydantic import BaseModel, Field

# pyowm imports - these were missing!
from pyowm.owm import OWM
from pyowm.utils.config import get_default_config

from langchain_core.tools import BaseTool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

import os

owm_key = os.environ['OWM_API_KEY']


def get_weather(city: str) -> dict:
    config_dict = get_default_config()
    config_dict["language"] = "en"

    owm = OWM(owm_key, config_dict)
    mgr = owm.weather_manager()

    observation = mgr.weather_at_place(city)
    weather = observation.weather

    return {
        "temperature": f"{weather.temperature('celsius')['temp']} °C",
        "humidity": f"{weather.humidity} %",
        "wind": f"{weather.wind()['speed']} m/s",
        "status": weather.detailed_status,
    }


class GetWeatherInput(BaseModel):
    city: str = Field(description="City name with country, e.g. 'Hyderabad, India'")


class GetWeatherTool(BaseTool):
    name: str = "get_weather"
    description: str = "Get current weather details for a city."
    args_schema: Type[BaseModel] = GetWeatherInput

    def _run(self, city: str) -> dict:
        return get_weather(city)

    async def _arun(self, city: str) -> dict:
        raise NotImplementedError("Async not supported yet")


# LLM
llm = make_llm()

# Tools
tools = [GetWeatherTool()]

# Agent
agent = create_agent(llm, tools)

# Simple memory
history: list = []


def run_agent(query: str) -> str:
    messages = history + [HumanMessage(content=query)]
    result = agent.invoke({"messages": messages})

    ai_response = result["messages"][-1].content

    history.append(HumanMessage(content=query))
    history.append(AIMessage(content=ai_response))

    return ai_response


# Multi-turn example
print(run_agent("What is the weather in Moscow, Russia?"))
print(run_agent("How does that compare to Amsterdam?"))
print(run_agent("Which city is warmer?"))

The weather in Moscow, Russia is currently **light rain**. Here are the details:

*   **Temperature:** 16.63 °C
*   **Humidity:** 68%
*   **Wind Speed:** 4.09 m/s


Here is how the weather in Amsterdam compares to Moscow:

*   **Temperature:** Amsterdam is significantly warmer at **21.49 °C**, compared to Moscow's **16.63 °C**.
*   **Humidity:** Amsterdam is slightly more humid (**57%**) than Moscow (**68%**).
*   **Wind:** The wind is stronger in Amsterdam (**6.17 m/s**) than in Moscow (**4.09 m/s**).
*   **Sky Condition:** In Amsterdam, the sky is overcast, whereas in Moscow it was light rain.


Based on the data provided, **Amsterdam** is warmer than Moscow.

*   **Amsterdam:** 21.49 °C
*   **Moscow:** 16.63 °C


In [10]:
result = agent.invoke({"messages": [("user", "Should I visit New York, USA this December based on the weather conditions?")]})
print(result["messages"][-1].content)

Based on the current weather data for New York, USA:

*   **Temperature:** 20.6°C (69°F)
*   **Conditions:** Few clouds
*   **Humidity:** 75%

**Verdict:** Yes, you should visit! The weather looks pleasant and mild for December. However, please note that this is the *current* forecast. Weather conditions can change quickly, so it's always a good idea to check the forecast again before your trip.


### Another OpenWeatherMap API

In [11]:
import json
from pydantic import BaseModel, Field
from pyowm.owm import OWM
from pyowm.utils.config import get_default_config

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

import os
owm_key = os.environ['OWM_API_KEY']

# ---- Weather tool ----
class WeatherInput(BaseModel):
    city: str = Field(description="City and country, e.g. 'Hyderabad, India'")


@tool("get_weather", args_schema=WeatherInput)
def get_weather(city: str) -> str:
    """Get current weather details for a city. Input must be 'City, Country'."""
    config_dict = get_default_config()
    config_dict["language"] = "en"

    owm = OWM(owm_key, config_dict)  # <-- your OWM key
    obs = owm.weather_manager().weather_at_place(city)
    w = obs.weather

    data = {
        "city": city,
        "temperature_c": w.temperature("celsius")["temp"],
        "humidity_pct": w.humidity,
        "wind_m_s": w.wind().get("speed"),
        "status": w.detailed_status,
    }
    return json.dumps(data, ensure_ascii=False)


# ---- LLM + Agent ----
llm = make_llm()

tools = [get_weather]
agent = create_agent(llm, tools)

# ---- Run ----
result = agent.invoke({"messages": [("user", "What is the weather in Merida, Mexico?")]})
print(result["messages"][-1].content)


The weather in Merida, Mexico is currently **overcast** with a temperature of approximately **27°C (80°F)**. The humidity is at 89%, and there is a light wind speed of about 2.57 m/s.


In [12]:
result = agent.invoke({"messages": [("user", "Should I visit Saint Peterburg, Russia this December based on the weather conditions?")]})
print(result["messages"][-1].content)

Based on the current weather data for Saint Petersburg, Russia:

*   **Temperature:** It is currently around **12°C (54°F)**. This is a mild but cool day.
*   **Conditions:** The sky is overcast with high humidity (76%).
*   **Wind:** There is a light breeze at 7 meters per second.

**Verdict:**
It might be pleasant enough for an outdoor visit, especially if you are looking for a walk or a short trip to the nearby beaches. However, given the high humidity and overcast skies, it could feel quite chilly and damp compared to typical summer weather. If you plan to stay overnight, you should bring a warm layer like a jacket or sweater.
